In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

from src.downloadData import download_dataset_csvs
summary = download_dataset_csvs(
    output_dir=repo_root /"data" /"raw",
    progress=True,
)

print(f"downloaded={len(summary.downloaded_paths)}, skipped={len(summary.skipped_paths)}")

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType

import os
import sys
import shutil

python_exe = sys.executable
os.environ["PYSPARK_PYTHON"] = python_exe
os.environ["PYSPARK_DRIVER_PYTHON"] = python_exe

spark = (
    SparkSession.builder
    .appName("EEG_Schizoprenia")
    .master("local[*]")
    .config("spark.pyspark.python", python_exe)
    .config("spark.pyspark.driver.python", python_exe)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark started:", spark.version)
print("Python:", sc.pythonExec)

In [ ]:
base_path = ".small/raw"
data_paths = list(Path(base_path).rglob("*/*.csv"))
print(f"Found {len(data_paths)} files. " \
      f"From `{data_paths[0]}` to `{data_paths[-1]}`")

In [ ]:
column_names = spark.read.csv(r".\data\raw\columnLabels.csv",
                              header=True).columns
print("List of column labels:")
for i, column in enumerate(column_names, start=1):
    print(f"  {i}. {column}" )

In [ ]:
df = spark.read.csv(r".small\raw\*\*.csv", 
                    inferSchema=True, # infer data types
                    header=False)
df = df.toDF(*column_names) 

row_count = df.count()
print(f"Read {row_count:,} records")

df.show(5)

In [ ]:
from pyspark.sql.functions import avg, max, min, percentile_approx

group_cols = ["subject", "trial", "condition"]
other_cols = [col_name for col_name in df.columns if col_name not in group_cols]
other_cols.remove("sample")

grp_functions = [(avg, "avg"),(max, "max"),(min, "min")]

trial_avg = df.groupBy(group_cols).agg(
                *[f(c).alias(f"{c}_{name}") for c in other_cols for f, name in grp_functions],
                *[percentile_approx(c, [0.25, 0.5, 0.75]).alias(f"{c}_quartiles") for c in other_cols]
            ).orderBy(group_cols)
trial_avg.show(15)

# In here I wanted to add mode as well but since mode is a heavy process, it caused memory errors

In [ ]:
trial_avg.coalesce(1).write.mode("overwrite").option("header", True).csv("data/processed/all_csv_avg")

In [ ]:
spark.stop()